# RC DeepSpatial：MERFISH 模块与机制实验
默认留出 1、2、3，seed 42。先训练 baseline/control 各 100 epochs，再从相同冻结 control 主干训练各残差臂 100 epochs。工程检查与真实效果结论分开；数据、旧项目和已有结果不改写。
更改物理 GPU 后重启内核；复用已有输出时仅在配置和源码完全相同时启用 RESUME。详细协议见项目 README。

In [ ]:
import os, sys
from pathlib import Path
GPU_ID = '0'
if 'torch' in sys.modules:
    raise RuntimeError('Restart the kernel before selecting a physical GPU.')
if GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
candidates = [Path.cwd(), Path.cwd() / 'RC_DeepSpatial', Path.cwd().parent, Path.cwd().parent / 'RC_DeepSpatial']
PROJECT = next((p.resolve() for p in candidates if (p / 'rcspatial').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Open this notebook from the RC_DeepSpatial project.')
sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT, '| physical GPU:', GPU_ID)

In [ ]:
import json
CONFIG = json.loads((PROJECT / 'configs/merfish.json').read_text(encoding='utf-8-sig'))
OUTPUT = PROJECT / 'outputs/merfish_v1'
HOLDOUTS = [1, 2, 3]
SEEDS = [42]  # Expand to [42, 0, 1, 2, 3] for optimization variability.
ARMS = ['baseline', 'control', 'R0', 'A', 'B', 'AB', 'ABC']
# Mechanism arms: ABC_zero, ABC_shuffle, ABC_single, ABC_no_geometry,
# ABC_no_anchor, AB_risk_only, AB_marginal_only.
STAGE = 'all'  # audit / base / train / generate / evaluate / all
RESUME = False
DEVICE = 'auto'
ROLE = 'development'
EXCLUDED_OUTER_TEST = []
print('Epochs per stage:', CONFIG['training']['epochs'], '| arms:', ARMS, '| output:', OUTPUT)

In [ ]:
from rcspatial.experiment import run, compare
records = run(CONFIG, OUTPUT, HOLDOUTS, SEEDS, ARMS, stage=STAGE, device=DEVICE, resume=RESUME,
              excluded=EXCLUDED_OUTER_TEST, role=ROLE)
print('Completed records:', len(records))
if STAGE in ('all', 'evaluate'):
    summary = compare(OUTPUT, allow_incomplete=True)
    display(summary)

机制分析先查看 shared/transport_local.csv、matching_support.csv，各臂 trajectory_sampling.csv、training_by_type.csv、marker_sparsification.csv，以及汇总 drift_to_control.csv。
rare 按可见训练丰度 ≤5% 定义，另报 ≤1%。model_class 与冻结表达 probe 分开评价，生成数量由可见来源决定。零支持细胞保留在输出与评价中。